[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Add ReID to Trackers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roboflow/trackers/blob/develop/docs/cookbooks/how-to-add-reid-to-trackers.ipynb)

BoT-SORT associates a detection with a track by how much their boxes overlap. When two people cross, the boxes overlap about equally and geometry alone picks the wrong one. Appearance ReID adds a second opinion: an encoder turns each crop into an embedding, and the appearance distance between a track and a detection breaks the tie.

This notebook enables ReID on BoT-SORT with the [`reid`](https://reid.roboflow.com/latest/) package, measures what it buys on MOT17 val-half against the same run without it, and then calibrates `appearance_threshold` on the data instead of inheriting it from a paper.

Runtime: about 30 minutes on a Colab T4, most of it encoding crops.

## Setup

### Check GPU availability

An encoder runs on every high-confidence detection in every frame, so a GPU is the difference between minutes and hours. If the cell below fails, switch the runtime type to GPU.

In [ ]:
!nvidia-smi

### Install dependencies

The `reid` extra pulls in the encoder package and matplotlib. `gdown` fetches the YOLOX detections from Google Drive.

In [ ]:
!pip install -q "trackers[reid]" gdown

### Imports

In [ ]:
import warnings
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import supervision as sv
import torch
from reid import FASTREID_MOT17_SBS50, ReIDModel

from trackers import BoTSORTTracker
from trackers.core.reid import (
    extract_ground_truth_embeddings,
    plot_appearance_distances,
    plot_frame_gap_sweep,
    sample_appearance_distances,
    sweep_frame_gap,
)
from trackers.eval import evaluate_mot_sequences
from trackers.io.frames import load_mot_frame_image
from trackers.io.mot import load_mot_file

warnings.filterwarnings("ignore")

device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()} | {device}")


## Download MOT17 val-half and YOLOX detections

Both runs share one set of detections, so any difference in the metrics comes from association rather than detection quality. The YOLOX detections are the ones the BoT-SORT paper used, which is what makes the numbers at the end comparable to published results.

In [ ]:
ROOT = Path("mot17-reid")
MOT17_VAL = ROOT / "mot17" / "val"
YOLOX_DIR = ROOT / "MOT17_yolox_dets"
YOLOX_ZIP = YOLOX_DIR / "yolox_detections_MOT17.zip"
YOLOX_GDRIVE_ID = "1BuXtPWf8QbPU_y1i2xY2IbTE-rj3l6qT"
OUTPUT_ROOT = ROOT / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

!trackers download --name mot17 --split val --asset annotations,frames --output {ROOT}

!mkdir -p {YOLOX_DIR}
!gdown {YOLOX_GDRIVE_ID} -O {YOLOX_ZIP}
!unzip -qo {YOLOX_ZIP} -d {YOLOX_DIR}

In [ ]:
VAL_SEQUENCES = [
    "MOT17-02-FRCNN",
    "MOT17-04-FRCNN",
    "MOT17-05-FRCNN",
    "MOT17-09-FRCNN",
    "MOT17-10-FRCNN",
    "MOT17-11-FRCNN",
    "MOT17-13-FRCNN",
]

SEQUENCES: dict[str, dict] = {}
for seq in VAL_SEQUENCES:
    ground_truth = MOT17_VAL / seq / "gt" / "gt.txt"
    images = MOT17_VAL / seq / "img1"
    detections = YOLOX_DIR / "val" / f"{seq.replace('-FRCNN', '')}_val.txt"
    if not (ground_truth.is_file() and images.is_dir() and detections.is_file()):
        print(f"  skip {seq}: missing gt, img1, or YOLOX detections")
        continue
    frames = sorted(images.glob("*.jpg"))
    SEQUENCES[seq] = {
        "ground_truth": ground_truth,
        "images": images,
        "detections": detections,
        "frames": frames,
    }
    print(f"  {seq}: {len(frames)} frames")

if not SEQUENCES:
    raise RuntimeError("No sequences ready, re-run the download cell above.")

SEQMAP = OUTPUT_ROOT / "MOT17-val.txt"
SEQMAP.write_text("name\n" + "\n".join(SEQUENCES) + "\n")
print(f"\n{len(SEQUENCES)} sequences ready")

## Load the ReID encoder

`fastreid_mot17_sbs50` was trained on MOT17, so it is the encoder to beat on this benchmark. Any other id from the `reid` package, or your own checkpoint, drops in here unchanged.

`appearance_threshold` is the distance above which BoT-SORT stops trusting appearance. It defaults to 0.25, the value the BoT-SORT paper uses. We start from 0.2 instead and check that choice against the data further down.

In [ ]:
REID_ENCODER = FASTREID_MOT17_SBS50
APPEARANCE_THRESHOLD = 0.2

reid_model = ReIDModel.from_pretrained(REID_ENCODER)
print(f"Encoder: {REID_ENCODER}  |  appearance_threshold: {APPEARANCE_THRESHOLD}")
print(reid_model.preprocessing.describe())

## Run BoT-SORT with and without ReID

Passing `reid_model` is the whole change. Everything else, detections included, is held fixed between the two runs.

`reid_ema_alpha` controls how much of a track's stored appearance survives each update: 0.9 keeps 90% of the running average, so a single blurry crop cannot overwrite a track's identity.

In [ ]:
def load_yolox_detections(path: Path) -> dict[int, sv.Detections]:
    """Load YOLOX detections (`frame,x1,y1,x2,y2,score`) keyed by 1-based frame index."""
    rows = []
    with path.open() as handle:
        for line in handle:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            frame, x1, y1, x2, y2, score = map(float, parts[:6])
            if score > 0:
                rows.append((int(frame), x1, y1, x2, y2, score))
    if not rows:
        return {}
    # Some files start at frame 0 and some at 1; align both to 1.
    offset = min(frame for frame, *_ in rows) - 1
    by_frame: dict[int, list[list[float]]] = {}
    for frame, x1, y1, x2, y2, score in rows:
        by_frame.setdefault(frame - max(offset, 0), []).append([x1, y1, x2, y2, score])
    return {
        frame: sv.Detections(
            xyxy=np.asarray(boxes, dtype=np.float32)[:, :4],
            confidence=np.asarray(boxes, dtype=np.float32)[:, 4],
        )
        for frame, boxes in by_frame.items()
    }


def write_mot_row(handle, frame_idx: int, detections: sv.Detections) -> None:
    """Append one frame of tracks as MOT rows: `frame,id,x,y,w,h,conf,-1,-1,-1`."""
    if detections.tracker_id is None:
        return
    for box, track_id in zip(detections.xyxy, detections.tracker_id):
        x1, y1, x2, y2 = box
        handle.write(f"{frame_idx},{int(track_id)},{x1:.2f},{y1:.2f},{x2 - x1:.2f},{y2 - y1:.2f},1,-1,-1,-1\n")


def run(name: str, build_tracker) -> object:
    """Track every sequence with a fresh tracker, then score the predictions."""
    prediction_dir = OUTPUT_ROOT / name / "preds"
    prediction_dir.mkdir(parents=True, exist_ok=True)
    for seq, spec in SEQUENCES.items():
        detections_by_frame = load_yolox_detections(spec["detections"])
        tracker = build_tracker()
        with (prediction_dir / f"{seq}.txt").open("w") as handle:
            for frame_idx, image_path in enumerate(spec["frames"], start=1):
                frame = cv2.imread(str(image_path))
                detections = detections_by_frame.get(frame_idx, sv.Detections.empty())
                tracked = tracker.update(detections, frame)
                if tracked.tracker_id is not None:
                    tracked = tracked[tracked.tracker_id != -1]
                write_mot_row(handle, frame_idx, tracked)
        print(f"  {name}: {seq} done")
    return evaluate_mot_sequences(
        gt_dir=MOT17_VAL,
        tracker_dir=prediction_dir,
        seqmap=SEQMAP,
        metrics=["CLEAR", "HOTA", "Identity"],
    )


baseline = run("botsort", lambda: BoTSORTTracker(enable_cmc=True))
with_reid = run(
    "botsort_reid",
    lambda: BoTSORTTracker(
        enable_cmc=True,
        reid_model=reid_model,
        reid_ema_alpha=0.9,
        appearance_threshold=APPEARANCE_THRESHOLD,
    ),
)

## Compare the two runs

IDF1 and AssA are the metrics to watch. Both reward keeping one identity on one person for the whole sequence, which is exactly what appearance is there to protect, while MOTA is dominated by detection quality and barely moves.

The reference row comes from a [MOT17 re-ID study](https://www-sop.inria.fr/members/Francois.Bremond/Postscript/Tomasz__SCCAI_2025.pdf) (Table 8 for HOTA, Table 13 for IDF1) that ran the same encoder, detections and threshold, so your uplift should land near theirs.

In [ ]:
STUDY_NO_REID = {"hota": 68.43, "idf1": 80.92}
STUDY_REID = {"hota": 68.95, "idf1": 81.98}


def metrics(result) -> tuple[float, float, float, float, int]:
    aggregate = result.aggregate
    return (
        aggregate.HOTA.HOTA * 100,
        aggregate.HOTA.AssA * 100,
        aggregate.CLEAR.MOTA * 100,
        aggregate.Identity.IDF1 * 100,
        aggregate.CLEAR.IDSW,
    )


print(f"{'Config':<26}  {'HOTA':>6}  {'AssA':>6}  {'MOTA':>6}  {'IDF1':>6}  {'IDSW':>5}")
print("-" * 68)
for label, result in (("BoT-SORT", baseline), ("BoT-SORT + ReID", with_reid)):
    hota, assa, mota, idf1, idsw = metrics(result)
    print(f"{label:<26}  {hota:6.2f}  {assa:6.2f}  {mota:6.2f}  {idf1:6.2f}  {idsw:5d}")

before, after = metrics(baseline), metrics(with_reid)
print(
    f"\nReID uplift:      HOTA {after[0] - before[0]:+.2f}   IDF1 {after[3] - before[3]:+.2f}   "
    f"IDSW {after[4] - before[4]:+d}"
)
print(
    f"Reference study:  HOTA {STUDY_REID['hota'] - STUDY_NO_REID['hota']:+.2f}   "
    f"IDF1 {STUDY_REID['idf1'] - STUDY_NO_REID['idf1']:+.2f}"
)

print(f"\n{'Sequence':<18}  {'HOTA':>6}  {'AssA':>6}  {'IDF1':>6}  {'IDSW':>5}")
print("-" * 50)
for seq in SEQUENCES:
    plain, reid = baseline.sequences[seq], with_reid.sequences[seq]
    print(
        f"{seq:<18}  {reid.HOTA.HOTA * 100:6.2f}  {reid.HOTA.AssA * 100:6.2f}  "
        f"{reid.Identity.IDF1 * 100:6.2f}  {reid.CLEAR.IDSW:5d}"
        f"   (IDSW without ReID: {plain.CLEAR.IDSW})"
    )

## Choose an appearance threshold

BoT-SORT gates appearance on `d_app = 0.5 * (1 - cosine_similarity)`, ignoring the appearance term whenever the distance exceeds `appearance_threshold`. Where to put that threshold depends on your encoder and your footage, so measure it rather than inherit it.

What matters is which pairs you measure. A tracker only ever compares crops from the same video within a few dozen frames of each other, so those are the pairs to sample: same-ID pairs it should accept, and different-ID pairs from the same window that could steal the match. `sample_appearance_distances` draws exactly those, giving every sequence an equal quota and every identity an equal chance, so one crowded sequence or one long track cannot decide the answer.

First, embed the ground-truth crops. This is the slow cell.

In [ ]:
# MOT17 marks pedestrians as class 1; confidence 0 rows are ignore-flagged.
embeddings, ids, frame_ids, sequence_ids = extract_ground_truth_embeddings(
    reid_model,
    MOT17_VAL,
    sequences=list(SEQUENCES),
    keep_classes=(1,),
)

print(f"pool: {len(embeddings)} crops, {len(np.unique(ids))} identities")


Now sample the pairs and look at the two distributions. A workable threshold sits in the valley between them: high enough to accept most same-ID pairs, low enough to reject the different-ID ones.

`rates_at` puts a number on that trade-off. There is no correct answer for both columns at once, and which way to lean depends on whether a lost track or a swapped ID hurts you more, which is why nothing here picks a threshold for you.

In [ ]:
distances = sample_appearance_distances(
    embeddings,
    ids,
    frame_ids,
    sequence_ids,
    same_id_pairs=5000,
    different_id_pairs=10000,
    minimum_frame_gap=1,
    maximum_frame_gap=30,  # the default lost_track_buffer, i.e. one second at 30 FPS
)

print(f"separability (ROC AUC): {distances.roc_auc:.3f}")
print(f"\n{'θ':>6}  {'same-ID accepted':>17}  {'different-ID accepted':>22}")
for threshold in (0.10, 0.20, 0.25, 0.30):
    same_id_rate, different_id_rate = distances.rates_at(threshold)
    print(f"{threshold:6.2f}  {same_id_rate:16.1%}  {different_id_rate:21.1%}")

plot_appearance_distances(
    distances,
    thresholds={APPEARANCE_THRESHOLD: "selected", 0.25: "default"},
    title=f"{REID_ENCODER} on MOT17 val ground truth",
)
plt.show()

## How far the threshold carries

The histogram above fixes the frame gap at 30, so it describes re-association within a second. It says nothing about finding a track again after a longer occlusion, which is the case appearance is supposed to rescue.

`sweep_frame_gap` repeats the sampling across widening gaps. Watch the same-ID band drift upward while the different-ID band stays put: identities become harder to recognise as time passes, but strangers do not become easier to confuse. A threshold tuned on adjacent frames therefore turns into a threshold that quietly refuses to re-find anything.

The lower panel reports ROC AUC, the chance that a random same-ID pair scores closer than a random different-ID pair. It summarises every possible threshold at once, so it separates "the encoder has degraded" from "our threshold is in the wrong place".

In [ ]:
sweep = sweep_frame_gap(embeddings, ids, frame_ids, sequence_ids, pairs_per_class=8000)

print(f"{'frame gap':>10}  {'ROC AUC':>8}  {'same-ID < θ':>12}  {'different-ID < θ':>17}")
for band in sweep:
    same_id_rate, different_id_rate = band.rates_at(APPEARANCE_THRESHOLD)
    print(f"{band.label:>10}  {band.roc_auc:8.3f}  {same_id_rate:11.1%}  {different_id_rate:16.1%}")

plot_frame_gap_sweep(
    sweep,
    thresholds={APPEARANCE_THRESHOLD: "selected", 0.25: "default"},
    title=f"{REID_ENCODER}: separability vs frame gap",
)
plt.show()

If you raise `lost_track_buffer` to recover tracks after long occlusions, raise `appearance_threshold` with it and re-read the different-ID column, otherwise the longer buffer buys nothing.

## Sample tracked frames

Finally, look at the output. Track ids should stay pinned to the same person across all four frames.

In [ ]:
VIZ_SEQ = next(iter(SEQUENCES))
VIZ_FRAMES = (1, 30, 60, 90)

predictions = load_mot_file(OUTPUT_ROOT / "botsort_reid" / "preds" / f"{VIZ_SEQ}.txt")
box_annotator = sv.BoxAnnotator(thickness=2, color_lookup=sv.ColorLookup.TRACK)
label_annotator = sv.LabelAnnotator(
    text_color=sv.Color.BLACK,
    text_scale=0.5,
    color_lookup=sv.ColorLookup.TRACK,
)

figure, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, frame_idx in zip(axes.ravel(), VIZ_FRAMES):
    frame = load_mot_frame_image(SEQUENCES[VIZ_SEQ]["images"], frame_idx)
    frame_data = predictions.get(frame_idx)
    scene = frame
    if frame_data is not None and len(frame_data.ids) > 0:
        detections = sv.Detections(
            xyxy=sv.xywh_to_xyxy(frame_data.boxes).astype(np.float32),
            tracker_id=frame_data.ids.astype(int),
        )
        scene = box_annotator.annotate(frame.copy(), detections)
        scene = label_annotator.annotate(
            scene, detections, labels=[str(int(track_id)) for track_id in detections.tracker_id]
        )
    ax.imshow(scene[:, :, ::-1])
    ax.set_title(f"{VIZ_SEQ}  frame {frame_idx}")
    ax.axis("off")

figure.suptitle("BoT-SORT + ReID", y=1.01)
figure.tight_layout()
plt.show()

## Next steps

You ran BoT-SORT with and without appearance ReID on MOT17, and calibrated the threshold on the pairs a tracker actually sees rather than on a paper's default.

To take this to your own footage: swap the encoder id for one trained closer to your domain, re-run the two threshold cells on a labeled slice of your data, and read the different-ID column before trusting the number. A pedestrian encoder on, say, soccer footage compresses every distance into a narrow band and needs a much tighter threshold, which the [ReID appearance guide](https://trackers.roboflow.com/latest/guides/reid/) walks through with SoccerNet numbers.